In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)


# Improved transfer: source-forest representations and nested few-shot adaptation

This experiment builds on `baseline.ipynb` and `preprocessing_age_labels.ipynb`. Its aim is to improve the fixed 50/50 source/target probability blend without spending additional Amsterdam labels or fitting on evaluation locations.

**Hypothesis:** the Madrid forest's four class probabilities discard useful structure and carry city-specific biases. Which leaf a pixel reaches in each source tree gives a richer nonlinear representation. A small regularized classifier can learn how that representation maps to Amsterdam age classes.

**Prespecified procedure:** compare 11 simple adapter recipes using inner spatial validation on the budget-limited adaptation sample; refit the selected recipe on that entire sample; evaluate once on its outer spatial fold. Candidates include the original fixed blend, target-only RF, and three regularization strengths for a leaf classifier alone or combined with source/target forests. A fixed 0.005 inner-F1 tie tolerance prefers earlier recipes, starting with the baseline.

**Important scope:** baseline outer-fold results were already viewed and motivated this experiment. The current selector never sees outer labels, but this is iterative development, not an untouched final test. The organizer's hidden test remains necessary for confirmation. Existing features, class boundaries, source model, five outer folds, label samples and three repetitions are preserved.

## 1. Inputs, versions and fixed candidate inventory

Read exact schemas and verify input hashes. Import the reusable, commented training procedure from `scripts/transfer_adaptation.py`. No new runtime dependencies are needed. The source forest stays frozen; its imputer and trees were trained only on Madrid.

The selection candidates are fixed here, before evaluating this experiment's outer scores. C controls the leaf classifier's L2 regularization: smaller values shrink the target-specific head more strongly. No parameter is chosen from the outer-fold comparisons below.


In [ ]:
from scripts.transfer_adaptation import (CLASSES, C_VALUES, RECIPES, probabilities,
                                         source_representation, fit_leaf_adapters, fit_adaptation, predict_adaptation, blend)
from pathlib import Path
import sys
import hashlib
import json
import time
import gc
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from IPython.display import display
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)
from threadpoolctl import threadpool_limits
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'pyproject.toml').is_file() and (p/'data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Start inside this project.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
INPUT = ROOT/'data'/'preprocessed_age_labels'
BASELINE = ROOT/'data'/'baseline'
OUTPUT = ROOT/'data'/'transfer_improved'
OUTPUT.mkdir(parents=True, exist_ok=True)
BASE_CONFIG = json.loads((BASELINE/'experiment.json').read_text())
FEATURES = BASE_CONFIG['feature_names']
N_REPETITIONS = BASE_CONFIG['repetitions']
CONFIG = BASE_CONFIG['preprocessing_configuration']
N_FOLDS = CONFIG['n_folds']
BUDGETS = CONFIG['budgets']
SEED = BASE_CONFIG['seed']
RF_PARAMS = BASE_CONFIG['rf_parameters']
TIE_TOLERANCE = .005


def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(
            lambda: stream.read(8*1024*1024), b''): digest.update(block)
    return digest.hexdigest()


baseline_manifest = json.loads((BASELINE/'manifest.json').read_text())
for name in ['madrid_source_model.joblib', 'validation_predictions.parquet', 'adaptation_membership.csv', 'summary.csv', 'fold_metrics.csv']:
    assert sha256(
        BASELINE/name) == baseline_manifest[name]['sha256'], f'Baseline artifact changed: {name}'
for name in ['configuration.json', 'amsterdam_supervised_features.parquet']:
    assert sha256(
        INPUT/name) == BASE_CONFIG['input_hashes'][name], f'Preprocessing changed: {name}'
SOURCE_PATH = BASELINE/'madrid_source_model.joblib'
SOURCE_HASH = sha256(SOURCE_PATH)
SOURCE = joblib.load(SOURCE_PATH)
AMSTERDAM = pd.read_parquet(INPUT/'amsterdam_supervised_features.parquet')
MEMBERSHIP = pd.read_csv(BASELINE/'adaptation_membership.csv')
OLD_SUMMARY = pd.read_csv(BASELINE/'summary.csv')
OLD_METRICS = pd.read_csv(BASELINE/'fold_metrics.csv')
OLD_PREDICTIONS = pd.read_parquet(BASELINE/'validation_predictions.parquet')
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__}
assert sklearn.__version__ == BASE_CONFIG['versions']['scikit_learn']
assert len(RECIPES) == 11 and set(FEATURES).isdisjoint(
    {'age_class', 'weighted_mean_year', 'fold', 'px_key', 'py_key', 'city'})
print(VERSIONS)
display(pd.DataFrame(RECIPES))

## 2. Why a richer source representation may help

These diagnostics are from the already completed baseline, not a new selection signal. A source-only model can make systematically different errors in a new city; blindly assigning half of every final vote to it may preserve those errors.

A source tree leaf is an integer ID specific to that tree, not an ordered numeric feature. One-hot encoding treats it as categorical. Leaves seen only in evaluation pixels are ignored by an encoder fitted on the adaptation-training sample. The learned head can recalibrate source partitions without retraining the Madrid forest or requiring raw imagery.


In [ ]:
old_classes = pd.read_csv(BASELINE/'class_summary.csv')
display(old_classes.loc[old_classes.strategy.eq('source_only'), [
        'age_class', 'precision', 'recall', 'f1']].round(4))
print('Prior baseline Amsterdam macro F1:')
display(OLD_SUMMARY.loc[OLD_SUMMARY.strategy.eq('source_target_blend'),
                        ['samples_per_class', 'macro_f1_mean', 'macro_f1_fold_sd']].round(4))
# This applies a frozen model only: no vocabulary or normalization is fitted on evaluation pixels.
LEAVES, SOURCE_PROB = source_representation(SOURCE, AMSTERDAM[FEATURES])
assert LEAVES.shape == (len(AMSTERDAM), len(
    SOURCE.named_steps['classifier'].estimators_))
assert AMSTERDAM.location_id.is_unique
assert AMSTERDAM.groupby('spatial_group').fold.nunique().max() == 1
POSITIONS = pd.Series(np.arange(len(AMSTERDAM)), index=AMSTERDAM.location_id)
print('Cached frozen source leaf representation:', LEAVES.shape)

## 3. Training and evaluation contract

`fit_adaptation` accepts **only the selected labeled sample**, its frozen source representations, its feature names and fixed hyperparameters. It never receives the outer validation frame. Three inner `StratifiedGroupKFold` splits keep spatial groups separate; if a training split lacks a class, the procedure tries two splits, then falls back to the original baseline if no valid inner split is possible.

For each inner training set, the target forest's imputer, the leaf encoder, and the leaf classifier are refitted. Unknown leaves at inner/outer evaluation are ignored. Each candidate is scored by mean inner-fold macro F1 including all four classes, even when an inner validation fold lacks a class. This can be noisy with five labels per class; diagnostics expose that instability.

After selection, target components are refitted on **all and only** the allocated labels. Exact saved samples are reused. A separate fixed C=1 leaf-only adapter is evaluated as an ablation, not used to change the primary selection procedure after seeing outer scores.


In [ ]:
def sample_for(fold, n, repetition):
    rows = MEMBERSHIP.loc[MEMBERSHIP.fold.eq(fold) & MEMBERSHIP.samples_per_class.eq(
        n) & MEMBERSHIP.repetition.eq(repetition)]
    index = POSITIONS.loc[rows.location_id].to_numpy()
    sample = AMSTERDAM.iloc[index].copy()
    evaluation = AMSTERDAM.loc[AMSTERDAM.fold.eq(fold)]
    assert len(sample) == 4*n and sample.location_id.is_unique
    assert sample.age_class.value_counts().reindex(CLASSES).eq(n).all()
    assert sample.fold.ne(fold).all()
    assert set(sample.spatial_group).isdisjoint(evaluation.spatial_group)
    assert set(sample.location_id).isdisjoint(evaluation.location_id)
    return sample, index


# Scientific edge case: validation-only leaves must not change a fitted vocabulary.
toy = np.array([[1, 10], [2, 20], [3, 30], [4, 40],
               [1, 20], [2, 30], [3, 40], [4, 10]])
heads = fit_leaf_adapters(toy, np.array([1, 2, 3, 4, 1, 2, 3, 4]), (1.,))
head = heads[1.]
before = [x.copy() for x in head.named_steps['onehot'].categories_]
unknown_prob = probabilities(head, np.array([[999, 999]]))
assert np.isfinite(unknown_prob).all() and np.allclose(
    unknown_prob.sum(axis=1), 1)
for a, b in zip(before, head.named_steps['onehot'].categories_):
    np.testing.assert_array_equal(a, b)
print('Unknown-leaf handling preserves the training vocabulary.')

## 4. Nested adaptation experiments

Run the same 75 combinations of five outer folds, five budgets and three sampling repetitions. Save inner-fold scores and the selected recipe, not just its resulting performance. A fixed C=1 leaf-only head shows whether source leaf features themselves are useful.

The baseline fixed blend is reconstructed using an identically seeded target forest and checked against its original saved predictions. This catches differences in sample order, preprocessing, class alignment or random state. Outer scores are printed for transparency but are never fed back into adapter selection.

Five example deployment bundles (fold 0 / repetition 0) are saved with the frozen source model's hash and relative path. They remain adaptations excluding outer fold 0, not models trained on those evaluation labels.


In [ ]:
metrics = []
class_metrics = []
confusions = []
prediction_parts = []
selection_parts = []
selected_rows = []


def record(strategy, fold, n, repetition, validation, prediction):
    y = validation.age_class.to_numpy(dtype=int)
    meta = {'strategy': strategy, 'fold': fold,
            'samples_per_class': n, 'repetition': repetition}
    metrics.append({**meta, 'macro_f1': f1_score(y, prediction, labels=CLASSES, average='macro', zero_division=0),
                    'accuracy': accuracy_score(y, prediction), 'balanced_accuracy': balanced_accuracy_score(y, prediction),
                    'n_validation': len(y)})
    p, r, f, s = precision_recall_fscore_support(
        y, prediction, labels=CLASSES, zero_division=0)
    for i, c in enumerate(CLASSES):
        class_metrics.append({**meta, 'age_class': int(c),
                             'precision': p[i], 'recall': r[i], 'f1': f[i], 'support': int(s[i])})
    counts = confusion_matrix(y, prediction, labels=CLASSES)
    for i, a in enumerate(CLASSES):
        for j, b in enumerate(CLASSES):
            confusions.append({**meta, 'true_class': int(a), 'predicted_class': int(b),
                               'count': int(counts[i, j]), 'row_proportion': counts[i, j]/counts[i].sum()})
    part = validation[['location_id', 'age_class']].rename(
        columns={'age_class': 'y_true'}).copy()
    part['y_pred'] = prediction
    for k, v in meta.items():
        part[k] = v
    prediction_parts.append(part)


baseline_predictions = OLD_PREDICTIONS.loc[OLD_PREDICTIONS.city.eq(
    'Amsterdam') & OLD_PREDICTIONS.strategy.eq('source_target_blend')]
for fold in range(N_FOLDS):
    eval_positions = np.flatnonzero(AMSTERDAM.fold.eq(fold).to_numpy())
    validation = AMSTERDAM.iloc[eval_positions]
    for repetition in range(N_REPETITIONS):
        for n in BUDGETS:
            start = time.perf_counter()
            sample, sample_positions = sample_for(fold, n, repetition)
            seed = SEED+10_000+100*fold+repetition
            artifact, inner_scores = fit_adaptation(sample, FEATURES, LEAVES[sample_positions], SOURCE_PROB[sample_positions],
                                                    RF_PARAMS, seed, tie_tolerance=TIE_TOLERANCE)
            target_prob = probabilities(
                artifact['target_pipeline'], validation[FEATURES])
            fixed_prob = .5*SOURCE_PROB[eval_positions]+.5*target_prob
            reconstructed = CLASSES[fixed_prob.argmax(axis=1)]
            previous = baseline_predictions.loc[baseline_predictions.fold.eq(fold) & baseline_predictions.repetition.eq(repetition) &
                                                baseline_predictions.samples_per_class.eq(n)].set_index('location_id').loc[validation.location_id]
            np.testing.assert_array_equal(
                reconstructed, previous.y_pred.to_numpy())
            with threadpool_limits(limits=1):
                leaf_prob = (probabilities(
                    artifact['leaf_pipeline'], LEAVES[eval_positions]) if artifact['leaf_pipeline'] is not None else None)
            chosen_prob = blend(
                artifact['recipe'], SOURCE_PROB[eval_positions], target_prob, leaf_prob)
            chosen_prediction = CLASSES[chosen_prob.argmax(axis=1)]
            record('nested_selected_transfer', fold, n,
                   repetition, validation, chosen_prediction)
            if artifact['recipe']['c'] == 1. and artifact['leaf_pipeline'] is not None:
                ablation_head = artifact['leaf_pipeline']
            else:
                ablation_head = fit_leaf_adapters(
                    LEAVES[sample_positions], sample.age_class.to_numpy(), (1.,))[1.]
            with threadpool_limits(limits=1):
                ablation_prob = probabilities(
                    ablation_head, LEAVES[eval_positions])
            record('source_leaf_c1_ablation', fold, n, repetition,
                   validation, CLASSES[ablation_prob.argmax(axis=1)])
            inner_scores['fold'] = fold
            inner_scores['samples_per_class'] = n
            inner_scores['repetition'] = repetition
            selection_parts.append(inner_scores)
            selected_rows.append({'fold': fold, 'samples_per_class': n, 'repetition': repetition,
                                  'recipe': artifact['recipe']['name'], 'inner_split_count': artifact['inner_split_count'],
                                  'fit_seconds': time.perf_counter()-start})
            if fold == 0 and repetition == 0:
                artifact.update({'source_model_file': '../baseline/madrid_source_model.joblib', 'source_sha256': SOURCE_HASH,
                                 'excluded_Amsterdam_fold': fold, 'samples_per_class': n, 'repetition': repetition})
                joblib.dump(artifact, OUTPUT /
                            f'adapter_fold0_n{n}_rep0.joblib', compress=3)
            print(
                f'Fold {fold}, repetition {repetition}, {n}/class: selected={artifact["recipe"]["name"]}, F1={metrics[-2]["macro_f1"]:.4f}; leaf-only={metrics[-1]["macro_f1"]:.4f}; {selected_rows[-1]["fit_seconds"]:.1f}s', flush=True)
            del artifact, ablation_head
    gc.collect()

## 5. Paired comparisons and selection diagnostics

Average repetitions within each outer fold before computing mean and sample SD across five folds. Compare with the original baseline on identical samples/evaluation rows; report the fold-wise paired F1 difference. SD of five fold differences is descriptive, not a confidence interval or a significance test.

Show every tested budget and both the prespecified selected procedure and the C=1 ablation. Do not choose a deployment recipe by looking for the largest outer-fold result. Candidate-selection frequencies and inner results explain whether the procedure relies on the source representation or falls back to the original blend.


In [ ]:
METRICS = pd.DataFrame(metrics)
CLASS_METRICS = pd.DataFrame(class_metrics)
CONFUSIONS = pd.DataFrame(confusions)
PREDICTIONS = pd.concat(prediction_parts, ignore_index=True)
SELECTION = pd.concat(selection_parts, ignore_index=True)
SELECTED = pd.DataFrame(selected_rows)
KEY = ['strategy', 'samples_per_class']
FOLD_MEANS = METRICS.groupby(
    KEY+['fold'], as_index=False)[['macro_f1', 'accuracy', 'balanced_accuracy']].mean()
SUMMARY = FOLD_MEANS.groupby(KEY, as_index=False).agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_fold_sd=('macro_f1', 'std'),
                                                      accuracy_mean=('accuracy', 'mean'), balanced_accuracy_mean=('balanced_accuracy', 'mean'), n_folds=('fold', 'nunique'))
old_fold = OLD_METRICS.loc[OLD_METRICS.strategy.eq('source_target_blend')].groupby(
    ['samples_per_class', 'fold'], as_index=False).macro_f1.mean().rename(columns={'macro_f1': 'baseline_f1'})
PAIRED = FOLD_MEANS.merge(
    old_fold, on=['samples_per_class', 'fold'], validate='many_to_one')
PAIRED['delta_f1'] = PAIRED.macro_f1-PAIRED.baseline_f1
DELTA = PAIRED.groupby(KEY, as_index=False).agg(delta_f1_mean=(
    'delta_f1', 'mean'), delta_f1_fold_sd=('delta_f1', 'std'))
COMPARISON = SUMMARY.merge(DELTA, on=KEY, validate='one_to_one')
display(COMPARISON.round(4))
print('Selected recipe frequencies:')
display(pd.crosstab(SELECTED.samples_per_class, SELECTED.recipe))
CLASS_SUMMARY = CLASS_METRICS.groupby(
    KEY+['age_class'], as_index=False)[['precision', 'recall', 'f1']].mean()
print('25/class class-level results:')
display(CLASS_SUMMARY.loc[CLASS_SUMMARY.samples_per_class.eq(25)].round(4))

## 6. Learning curves and class confusion

Plot the original fixed blend, the nested selection procedure and the leaf-only ablation together. Error bars are SD across five outer fold means. The x-axis uses log2 spacing. Confusion matrices average row proportions within folds/repetitions before averaging outer folds, so long evaluation folds do not dominate.


In [ ]:
plt.rcParams.update(
    {'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(10, 5))
old = OLD_SUMMARY.loc[OLD_SUMMARY.strategy.eq(
    'source_target_blend')].sort_values('samples_per_class')
ax.errorbar(np.log2(old.samples_per_class), old.macro_f1_mean,
            yerr=old.macro_f1_fold_sd, marker='o', capsize=3, label='Original 50/50 blend')
for strategy, label in [('nested_selected_transfer', 'Nested selected transfer'), ('source_leaf_c1_ablation', 'Source leaf representation, C=1')]:
    sub = SUMMARY.loc[SUMMARY.strategy.eq(
        strategy)].sort_values('samples_per_class')
    ax.errorbar(np.log2(sub.samples_per_class), sub.macro_f1_mean,
                yerr=sub.macro_f1_fold_sd, marker='o', capsize=3, label=label)
ax.set_xticks(np.log2(BUDGETS), [str(n) for n in BUDGETS])
ax.set(xlabel='Amsterdam labeled locations per class (log2 spacing)', ylabel='Macro F1',
       ylim=(0, 1), title='Transfer comparison: five spatial outer folds')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT/'transfer_learning_curve.png', bbox_inches='tight')
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, strategy in zip(axes, ['nested_selected_transfer', 'source_leaf_c1_ablation']):
    sub = CONFUSIONS.loc[CONFUSIONS.strategy.eq(
        strategy) & CONFUSIONS.samples_per_class.eq(25)]
    matrix = sub.groupby(['fold', 'true_class', 'predicted_class']).row_proportion.mean().groupby(
        ['true_class', 'predicted_class']).mean().unstack().reindex(index=CLASSES, columns=CLASSES)
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap='Blues')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{matrix.iloc[i, j]:.2f}', ha='center', va='center',
                    color='white' if matrix.iloc[i, j] > .5 else 'black')
    ax.set_xticks(range(4), CLASSES)
    ax.set_yticks(range(4), CLASSES)
    ax.set(title=strategy+' (25/class)',
           xlabel='Predicted class', ylabel='True class')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=.8)
fig.savefig(OUTPUT/'transfer_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7. Verify predictions, selection and persisted adapters

Recompute each F1 from its validation predictions. Check run counts and that every selected candidate was chosen using its own inner scores under the documented tie rule. Verify the same budget-limited locations are stored in the persisted models and compare their reloaded predictions with the recorded outer results. Confirm the source artifact has not changed.

These checks complement the inner split/group assertions in the training helper. No outer labels or additional target labels are supplied to `fit_adaptation`, and no outer-feature distribution is fitted by the source transform or leaf encoder.


In [ ]:
RUN = ['strategy', 'samples_per_class', 'fold', 'repetition']
assert len(METRICS) == 2*N_FOLDS*N_REPETITIONS*len(BUDGETS)
assert len(SELECTED) == N_FOLDS*N_REPETITIONS*len(BUDGETS)
assert not PREDICTIONS.duplicated(RUN+['location_id']).any()
for key, part in PREDICTIONS.groupby(RUN):
    score = f1_score(part.y_true, part.y_pred, labels=CLASSES,
                     average='macro', zero_division=0)
    row = METRICS
    for column, value in zip(RUN, key):
        row = row.loc[row[column].eq(value)]
    assert len(row) == 1 and np.isclose(row.macro_f1.iloc[0], score)
for _, chosen in SELECTED.iterrows():
    scores = SELECTION.loc[SELECTION.fold.eq(chosen.fold) & SELECTION.repetition.eq(
        chosen.repetition) & SELECTION.samples_per_class.eq(chosen.samples_per_class)]
    if len(scores):
        means = scores.groupby('candidate').macro_f1.mean()
        expected = next(
            r['name'] for r in RECIPES if means[r['name']] >= means.max()-TIE_TOLERANCE)
        assert chosen.recipe == expected
    else:
        assert chosen.recipe == RECIPES[0]['name']
for n in BUDGETS:
    artifact = joblib.load(OUTPUT/f'adapter_fold0_n{n}_rep0.joblib')
    sample, _ = sample_for(0, n, 0)
    assert set(artifact['adaptation_location_ids']) == set(sample.location_id)
    assert artifact['selection_used_outer_labels'] is False
    evaluation = AMSTERDAM.loc[AMSTERDAM.fold.eq(0)]
    prediction, prob = predict_adaptation(artifact, SOURCE, evaluation)
    saved = PREDICTIONS.loc[PREDICTIONS.strategy.eq('nested_selected_transfer') & PREDICTIONS.fold.eq(0) &
                            PREDICTIONS.repetition.eq(0) & PREDICTIONS.samples_per_class.eq(n)].set_index('location_id')
    np.testing.assert_array_equal(
        prediction, saved.loc[evaluation.location_id, 'y_pred'].to_numpy())
assert sha256(SOURCE_PATH) == SOURCE_HASH
print('Verified all F1 scores, inner selection choices, identical baseline reconstructions and five persisted adapters; source artifact unchanged.')

## 8. Export artifacts and summarize the actual outcome

Keep this experiment separate from baseline artifacts. Save all outer predictions, class metrics, confusion matrices, paired differences, selected recipes, full inner scores, exact input/code hashes and software versions. The generated Markdown report states gains or regressions at every budget rather than claiming uniform improvement.

The fixed source forest is a valid Stage 1 model. The parameterized `fit_adaptation` procedure plus its input schema provides a reproducible Stage 2 approach. For a genuinely new city, supply only the allowed local labeled sample and its source representations; do not pass a hidden test set. The saved example bundles exclude Amsterdam outer fold 0.


In [ ]:
for name, frame in [('fold_metrics', METRICS), ('summary', SUMMARY), ('comparison', COMPARISON), ('paired_fold_differences', PAIRED),
                    ('class_metrics', CLASS_METRICS), ('class_summary',
                                                       CLASS_SUMMARY), ('confusion_matrices', CONFUSIONS),
                    ('selected_recipes', SELECTED), ('inner_selection_scores', SELECTION)]:
    frame.to_csv(OUTPUT/f'{name}.csv', index=False)
PREDICTIONS.to_parquet(OUTPUT/'validation_predictions.parquet', index=False)
provenance = {'seed': SEED, 'repetitions': N_REPETITIONS, 'features': FEATURES, 'candidate_recipes': RECIPES,
              'inner_splits_requested': 3, 'inner_split_fallback': [2, 'fixed baseline'], 'tie_tolerance': TIE_TOLERANCE,
              'source_model_sha256': SOURCE_HASH, 'script_sha256': sha256(ROOT/'scripts'/'transfer_adaptation.py'),
              'baseline_input_hashes': {name: sha256(BASELINE/name) for name in ['experiment.json', 'summary.csv', 'fold_metrics.csv', 'adaptation_membership.csv', 'validation_predictions.parquet']},
              'target_feature_sha256': sha256(INPUT/'amsterdam_supervised_features.parquet'), 'versions': VERSIONS,
              'outer_scores_used_for_current_selector': False, 'baseline_outer_results_previously_viewed': True,
              'organizer_test_sets_used': False, 'new_amsterdam_labels_used': 0}
(OUTPUT/'experiment.json').write_text(json.dumps(provenance, indent=2)+'\n')


def fmt(row): return f'{row.macro_f1_mean:.4f} ± {row.macro_f1_fold_sd:.4f}'


lines = ['# Source-representation transfer experiment', '',
         'Frozen Madrid forest leaves are one-hot encoded and used by a regularized Amsterdam classifier. Eleven recipes are selected using spatial inner validation restricted to each permitted label budget; the selected components are then refitted on those same labels. No additional Amsterdam labels or external datasets are used.', '',
         '| Labels/class | Original blend | Nested selected transfer | Paired Δ macro F1 | Leaf-only C=1 ablation |',
         '| ---: | ---: | ---: | ---: | ---: |']
for n in BUDGETS:
    old = OLD_SUMMARY.loc[OLD_SUMMARY.strategy.eq(
        'source_target_blend') & OLD_SUMMARY.samples_per_class.eq(n)].iloc[0]
    selected = COMPARISON.loc[COMPARISON.strategy.eq(
        'nested_selected_transfer') & COMPARISON.samples_per_class.eq(n)].iloc[0]
    ablation = COMPARISON.loc[COMPARISON.strategy.eq(
        'source_leaf_c1_ablation') & COMPARISON.samples_per_class.eq(n)].iloc[0]
    lines.append(
        f'| {n} | {fmt(old)} | {fmt(selected)} | {selected.delta_f1_mean:+.4f} | {fmt(ablation)} |')
primary = COMPARISON.loc[COMPARISON.strategy.eq('nested_selected_transfer')]
lines += ['', f'Average F1 across the five budgets: {primary.macro_f1_mean.mean():.4f}; average paired change: {primary.delta_f1_mean.mean():+.4f}.',
          f'Budgets with positive mean paired change: {int(primary.delta_f1_mean.gt(0).sum())} of {len(BUDGETS)}.', '',
          '## Validation and limitations', '',
          '- Five outer spatial folds, three identical seeded label-sampling repetitions per fold. Means average repetitions within a fold first; SD is across five fold means, not a confidence interval.',
          '- Inner grouped validation uses only the 5/25/50/100/200 labels per class. It refits imputation, leaf vocabularies and heads inside each inner training split. No evaluation labels or evaluation-feature statistics are fitted.',
          '- Candidate choices are based on mean inner macro F1, with a predefined 0.005 tie tolerance favoring earlier candidates. Diagnostics are saved, including unstable low-data selections and fallbacks.',
          '- Reconstructed original-blend predictions match every original evaluation run, verifying that comparisons reuse exactly the same samples and random states.',
          '- The C=1 leaf-only result is a prespecified ablation. It is not used retrospectively to choose a winner or change the nested selector.',
          '- The baseline scores had already been viewed and informed this follow-up design. These are iterative development results, not a new independent test; confirm any gains on organizer-held-out data.',
          '- Class boundaries remain city-specific; block size is in coordinate-key units and neighboring blocks can touch. Existing label and spatial-independence limitations remain.', '',
          '## Reproduction and artifacts', '',
          'Run `preprocessing_age_labels.ipynb`, `baseline.ipynb`, then `transfer_improved.ipynb`. The callable training/inference code is `scripts/transfer_adaptation.py`. Generated artifacts are in `data/transfer_improved/`, including five example fold-0 adapters, inner scores, selected recipes, class metrics, predictions, charts, input hashes and exact settings. The source model is reused from `data/baseline/madrid_source_model.joblib`. No runtime dependencies changed.', '',
          ]
(ROOT/'reports'/'TRANSFER_RESULTS.md').write_text('\n'.join(lines)+'\n')
pd.testing.assert_frame_equal(PREDICTIONS, pd.read_parquet(
    OUTPUT/'validation_predictions.parquet'))
pd.testing.assert_frame_equal(COMPARISON, pd.read_csv(
    OUTPUT/'comparison.csv'), check_exact=False, rtol=1e-12, atol=1e-12)
manifest = {p.name: {'bytes': p.stat().st_size, 'sha256': sha256(p)} for p in sorted(
    OUTPUT.iterdir()) if p.is_file() and p.name != 'manifest.json'}
(OUTPUT/'manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
print('Verified artifact round trips. Report:',
      ROOT/'reports'/'TRANSFER_RESULTS.md')
display(COMPARISON.loc[COMPARISON.strategy.eq(
        'nested_selected_transfer')].round(4))

## Interpretation and next decisions

Use the paired budget table above to judge what improved and where it failed. Inner validation can be noisy at five labels per class; a richer representation does not guarantee better transfer. The selected procedure may legitimately return the old baseline in some runs. Do not drop those runs from averages.

A future revision could evaluate fewer/more conservative recipes or a different learned representation, but it should be a separately documented experiment with fixed inner-selection rules. Increasing the label budget to tune a few-shot model would invalidate the task comparison. Unlabeled target-data alignment would need a clearly declared inductive/transductive protocol; this experiment fits nothing on outer target features.